In [2]:
from proactvl.infer.multi_assistant_inference import MultiAssistantStreamInference

/home/v-weicaiyan/miniconda3/envs/proactvl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/v-weicaiyan/miniconda3/envs/proactvl/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
[2026-02-04 07:28:03,705] - [INFO] - [qwen_omni_utils.v2_5.vision_process]: set VIDEO_TOTAL_PIXELS: 90316800


# Config

In [ ]:
# config
ckpt_path = ''
model_config = None
infer_config = {
    'max_kv_tokens': 16384,
    'assistant_num': 2, # assistant number, initialize two assistants
    'enable_tts': False,
    'state_threshold': 0.7,
}
generate_config = {
    'do_sample': True,
    'max_new_tokens': 12,
    'temperature': 0.7,
    'top_p': 0.9,
    'repetition_penalty': 1.15,
}
talker_config = None
device_id = 0


# Load Model

In [4]:
# load model
stream_infer = MultiAssistantStreamInference(model_config, ckpt_path, infer_config, generate_config, talker_config, f'cuda:{device_id}')

Loading model from checkpoint: oaaoaa/proactvl_base_liveccbase


Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 42.11it/s]


[OK] Added special tokens to tokenizer.


Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 52.61it/s]


In [5]:
# set system prompt 
system_prompt = ('You are a live commentator for a League of Legends (LoL) match. '
'Your role is to independently analyze and narrate the game, delivering insightful, engaging, and natural commentary just like a human expert.')
stream_infer.assistants[0].prime_system_prompt(system_prompt)
system_prompt_2 = 'You are a League of Legends play-by-play assistant, taking the role of supporting the main commentator during match commentary.'
stream_infer.assistants[1].prime_system_prompt(system_prompt_2)

[Assistant] setting system prompt of assistant 0 to:
You are a live commentator for a League of Legends (LoL) match. Your role is to independently analyze and narrate the game, delivering insightful, engaging, and natural commentary just like a human expert.


[Assistant] setting system prompt of assistant 1 to:
You are a League of Legends play-by-play assistant, taking the role of supporting the main commentator during match commentary.


# Load Video

In [6]:
video_path = './asset/sample.mp4'
video_begin = 0
video_end = 60
duration = video_end - video_begin
stream_infer.register_video_reader(video_path, video_begin, video_end)

Reading video ./asset/sample.mp4 from 0 to 60
Qwen2VLProcessor


qwen-vl-utils using torchvision to read video.
/home/v-weicaiyan/miniconda3/envs/proactvl/lib/python3.10/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


In [7]:
overall_cc = {}
assistant_responses = None
for t in range(duration):
    current_second = video_begin + t
    history = ''
    user_query = ''
    assistant_responses, _ = stream_infer.infer_one_chunk(current_second, history=history, user_query=user_query, previous_responses=assistant_responses)
    for assistant_id in assistant_responses:
        assistant_response = assistant_responses[assistant_id]
        if assistant_response.active:
            commentary = assistant_response.commentary.strip()
            overall_cc[current_second] = commentary if assistant_response is not None else ''
            print(f'[Sec: {current_second}({assistant_response.score})] Assistant {assistant_id}: {commentary}')
        else:
            print(f'[Sec: {current_second}({assistant_response.score})] Assistant {assistant_id}: <|SILENCE|>')

# print('Final Commentary:', overall_cc)



Assistant 0 history prompt: 


Assistant 1 history prompt: 
[Sec: 0(0.6015625)] Assistant 0: The big difference here is ...
[Sec: 0(0.5703125)] Assistant 1: <|SILENCE|>
Assistant 0 history prompt: 
1, AssistantResponse(assistant_id=1, assistant_name='SPEAKER_1', commentary=' ...', active=False, score=0.5703125, begin_second=0)
Assistant 1 history prompt: 
0, AssistantResponse(assistant_id=0, assistant_name='SPEAKER_0', commentary=' The big difference here is ...', active=True, score=0.6015625, begin_second=0)
[Sec: 1(0.9453125)] Assistant 0: the amount of CC available ...
[Sec: 1(0.08740234375)] Assistant 1: <|SILENCE|>
Assistant 0 history prompt: 
1, AssistantResponse(assistant_id=1, assistant_name='SPEAKER_1', commentary=' ...', active=False, score=0.08740234375, begin_second=1)
Assistant 1 history prompt: 
0, AssistantResponse(assistant_id=0, assistant_name='SPEAKER_0', commentary=' the amount of CC available ...', active=True, score=0.9453125, begin_second=1)
[Sec: 2(0.9921875)] Assistant 0: for both of these ..